# 0.3 · The LGD prior

*0. General · notebook 0.3 of the story.* ← [0.2 · the PD prior](<0.2_prior_visualisation_pd.ipynb>) · [1.1 · PD training](<../1. Experiment 1/1.1_pd_training.ipynb>) →

**What does our credit prior generate for LGD, how does it differ from TabICL's own, and does it
look like the real books of 0.1?** Experiment 1 then trains on both and lets the benchmark decide;
this notebook checks beforehand that the prior produces what it claims.

- **Original** is the prior at `credit_fraction = 0` — TabICLv2's `graph_scm`, unchanged, and
  Experiment 1's control. Its regression target is standard-scaled, so it is not on [0, 1] at all.
  **Credit** is ours, at `credit_fraction = 1`.
- Both are drawn live from `config/Exp1_LGD.yaml` through `TaskGenerator`, the code path training
  uses, with the settings of the sweep's first arm: filter mode `tabicl` and the mild boundary range.
  Every generated table has 1,024 rows and is zero-padded to 100 columns.
- **A caveat on reproducibility.** Our prior's LGD targets reproduce exactly from run to run, but
  the original prior's tasks change even at a fixed seed (the upstream graph code orders its
  feature groups by Python's per-process string hash; 0.2 has the details), and so does the column
  order of ours (our generator shuffles columns with torch's global random generator, which is not
  seeded here). Statements about the original prior are therefore quoted to the precision that
  survives a rerun.
- **How our LGD target is built in this experiment.** The config runs the target in `quantile`
  mode: the share of rows at exactly 0 and the share at exactly 1 are each drawn uniformly from
  `boundary_mass_range` — [0.02, 0.30] mild, [0.15, 0.60] aggressive, the total capped at 0.60 — and
  the interior follows a Kumaraswamy curve whose two shapes are drawn log-uniformly from [0.3, 4.0].
  The mapping is applied to the ranks of the causal graph's output, so the target stays a monotone
  function of its signal. The collateral, workout and segment loss stories in the code belong to
  `mechanism` mode, which this experiment does not use. (The config's `interior_shape_range` key is
  not read by the sampler — `sample_lgd_shape` reads `shape_ab_range`, whose default [0.3, 4.0] is
  what runs.)

**The order.** **A** puts the target of both priors against the real books: single tasks, where the
boundary mass sits, and how far each whole distribution is from real data. **B** takes the credit
mechanisms one at a time: boundary intensity, distribution shift, informative missingness. **C** judges
the task as a whole: its predictability next to real data (and what the filter keeps), one table, and
how its features depend on each other.

## Grounding in the literature

LGD is regression on a [0, 1] target with point masses at both ends. TabICLv2 standard-scales a
regression target — an affine map that keeps its shape but not its [0, 1] support — and produces ties
at the extremes only by accident, through the ±4 SD outlier clamp (`repositories/TabICL.txt`
`outlier_removing`). Our prior builds the atoms on purpose. Representing them is not the obstacle:
the regression head predicts 999 quantiles (`papers/2026/02_Qu_TabICLv2` §I), which can express a
point mass exactly, and the base prior already carries a Kumaraswamy [0, 1] warp
(`repositories/NanoTabICL.txt` `rand_kumaraswamy_act`). O'Prior supports regression but evaluates
only classification (`papers/2026/05_Bouadi_ShapingThePrior`), so bounded-target priors are open
ground; the primary metrics are distributional, because CRPS scores the whole predictive distribution
(`SYNTHESIS.md`). Pin `e5ce016`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib
ROOT = pathlib.Path.cwd()
# Walk up to the repository root — the notebook may be opened from its chapter folder, from
# notebooks/, or from the root — then work FROM the root, so relative paths (config/...) resolve
# exactly as under `python -m src.utils.run_notebooks`.
while not (ROOT / "src" / "visualize").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.visualize import exp1_plots as e1, figures, pool_plots as pp, mechanism_plots, prior_plots, style, summaries, literature

style.apply()
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK   = "lgd"
CONFIG = "config/Exp1_LGD.yaml"   # Exp1 is the prior sweep, so it is the one to visualise
N      = 500     # tasks drawn per prior

# Clears THIS notebook's figure folder — and no other — before anything is drawn.
FIGS = figures.FigureSaver("0.3_prior_visualisation_lgd")

## Loading the two priors and the real data

Pre-generated pools when they are on disk, otherwise 500 tasks per prior generated live — the printed
`source` says which. The real LGD datasets of 0.1 are loaded alongside, so that every
comparison below is against them.

In [ ]:
variants = pp.discover_pools(TASK)
print("pools found:", ", ".join(variants) if variants else "none - generating live")
loaded, SOURCE = pp.load_variants_or_generate(TASK, n=N, seed=0, config=CONFIG)
print("source =", SOURCE, "|", {k: len(v) for k, v in loaded.items()})

REAL = summaries.load_real_datasets(TASK)
print(f"real {TASK.upper()} datasets loaded: {len(REAL)}")
REFERENCE = pp.real_reference(TASK)

## A · Does the target look like real LGD data?

### A1 · The two priors in numbers

Per prior: the table size (rows are always 1,024; features counts only the columns that vary), the
share of tasks whose target lies in [0, 1], and the boundary mass — mean, 10th and 90th percentile,
and the share of tasks with any atom.

In [ ]:
pp.variant_summary(loaded, TASK)

### A2 · What does a single task's target look like?

**What it shows.** The target histogram of five tasks per prior, the same draw in each column: the share of rows per bin on the y axis, the target on the x axis — the original prior's on its own standardised scale (its minimum and maximum as ticks), ours on [0, 1].

**Why it matters.** A single task is what the model has to predict in context. Before any average, this shows whether a task is bounded, where its atoms are and what lies between them.

**What it says.** The original targets are standardised, each on its own scale, and their shapes range from bell-shaped to multimodal to nearly constant; some draws take only a handful of values, and atoms appear only as ties. (Which five original tasks appear changes between runs.) Every credit target lies on [0, 1] with spikes at 0 and 1 of different heights and an interior that is U-shaped, rising or falling from task to task.

In [ ]:
FIGS.save(pp.plot_target_shapes_by_variant(loaded), "target_shapes_by_variant",
    caption="Histograms of the target of five single generated tasks per prior, share of rows per bin, 25 bins; the original prior's target on its own standardised scale with its minimum and maximum as ticks, the credit prior's on [0, 1]; columns show the same draw index.");

### A3 · Where does the boundary mass sit?

**What it shows.** For every task, the share of rows at the target's minimum against the share at its maximum — for our prior and the real books that is at exactly 0 and exactly 1; for the original prior at its own extremes. Stars are the seven real datasets; on the dotted diagonal both atoms are equal.

**Why it matters.** Total boundary mass hides the direction (0.1 A3): a recovery-heavy book and a loss-heavy one can share a total. A prior should cover the real books in both coordinates.

**What it says.** Most original tasks sit at the origin — no ties at all — and a few lie far out: targets that take only a handful of values, like some draws in A2. Ours fill the square [0.02, 0.30] × [0.02, 0.30], exactly the mild range. Six of the seven real books lie inside that square; `heloc`, with 21.6 % at 0 and 51.5 % at 1, lies above it — only the aggressive range [0.15, 0.60] reaches its loss atom.

In [ ]:
FIGS.save(e1.plot_boundary_mass_sources(loaded, REAL), "boundary_mass_sources",
    caption="Share of rows at the target minimum against share at the maximum, one point per generated task and one panel per prior, stars for the seven real LGD datasets and the dotted diagonal for equal mass at both ends.");

### A4 · How far is each prior's target from the real ones?

**What it shows.** For each prior, the total-variation distance between its pooled target distribution (40 fixed bins on [0, 1]) and each real dataset's: one dot per real dataset, the diamond their mean; 0 is identical, 1 disjoint. The original prior's standardised targets are min-max scaled onto [0, 1] first, task by task.

**Why it matters.** One number per prior for how close its whole target distribution comes — atoms and interior together — and, through the dots, whether it is close to some books or to all. O'Prior's generator supports regression but evaluates only classification (`papers/2026/05_Bouadi_ShapingThePrior`), so there is no published yardstick for this.

**What it says.** The two are close on average — about 0.34 for ours against 0.35 — but differ in where they are close. Ours stays between 0.26 and 0.46 from every real book; the original prior, rescaled, comes very close to `lgd_freddie` (about 0.1 or less), whose flat interior it resembles, and far from `heloc` (about 0.6), whose atoms it lacks. Realism is a sanity check, not the claim — O'Prior finds mechanism diversity, not realism, drives transfer.

In [ ]:
FIGS.save(e1.plot_prior_realism_ranking(loaded, REAL, task=TASK), "prior_realism_ranking",
    caption="Total-variation distance between each prior's pooled target distribution and each real LGD dataset's over 40 fixed bins on [0, 1], one dot per real dataset and a diamond at the mean; the original prior's targets are min-max scaled to [0, 1] per task.");

## B · The credit mechanisms, one at a time

Each figure switches one mechanism of our prior on, generated live from the same config.

### B1 · How heavy are the boundary atoms?

**What it shows.** The target pooled over 50 tasks, as the share of rows per bin, for the original prior and for ours at the two intensities Experiment 1 sweeps; each panel gives the mean share of rows at the boundaries.

**Why it matters.** Boundary intensity is Experiment 1's LGD intensity lever: how much of each task's mass the prior places on the atoms. The real books span 1.8 % to 73 % (0.1 A3).

**What it says.** The original prior's pooled target is a bell on its standardised scale; roughly a tenth of its rows are tied at a task's minimum or maximum. Ours puts about 31 % of the rows at 0 or 1 in the mild setting and about 58 % in the aggressive one, the interior shrinking to a thin floor between the spikes.

In [ ]:
FIGS.save(mechanism_plots.intensity_atoms(CONFIG, n=50), "intensity_atoms",
    caption="Histogram of the target pooled over 50 generated tasks, share of rows per bin, for the original prior and the credit prior at mild and at aggressive boundary intensity; each panel states the mean share of rows at the boundaries.");

### B2 · Distribution shift — does the query look like the context?

**What it shows.** For each kind of shift, one point per generated task: the mean LGD of its context rows (the rows the model conditions on) against that of its query rows (the rows it must predict). Blue tasks have the shift switched on; grey ones are the same prior without it. On the dashed line the two parts are alike. For the covariate shift the axes show one feature, in standard deviations.

**Why it matters.** Losses scored today come from a later cohort or a different mix than the ones the model conditions on, and TFMs lose ground off the i.i.d. regime (`papers/2026/06_Purucker_BeyondIID`).

**What it says.** The covariate shift is strong: the context covers one end of a feature and the query the other. The prior-probability shift moves the query's mean LGD by about 0.31 on average, up or down about equally often. The cohort shift cannot be told apart from no shift (a mean gap of 0.076 against 0.090 without it) — expected in `quantile` mode, which generates no vintage blocks, so the cohort split is an ordinary random split.

In [ ]:
FIGS.save(mechanism_plots.shift_kinds(CONFIG, n=60), "shift_kinds",
    caption="Context against query mean per generated task, one panel per shift kind (cohort, covariate, prior probability), 40 tasks with the shift switched on and 40 without; mean LGD for the target shifts, the shifted feature in standard deviations for the covariate shift, and the dashed line where query equals context.");

### B3 · Informative missingness — is a missing value a signal?

**What it shows.** The share of values missing in each sixth of the LGD outcome, when missing values are placed completely at random (coupling 0) and when their probability rises with the outcome (coupling 2), on a controlled table of 4,000 rows with 20 % missing.

**Why it matters.** A missing value can carry the answer. TabICLv2 mean-imputes it away (`repositories/TabICL.txt` `TransformToNumerical`); the Exp1 config couples missingness to the outcome (coupling 1.0 on 30 % of the columns, 5–35 % missing each) and adds a was-missing indicator column.

**What it says.** At coupling 2 the missing share climbs from about 5 % in the lowest-loss sixth to about 57 % in the highest; at random it stays at 20 % throughout.

In [ ]:
FIGS.save(mechanism_plots.informative_missingness(TASK), "informative_missingness",
    caption="Share of values missing against the LGD outcome, sextile midpoints, on a controlled table of 4,000 rows, under missingness completely at random (coupling 0) and missingness tied to the outcome (coupling 2).");

## C · Is it the right task?

### C1 · How predictable are the tasks, next to real data?

**What it shows.** Every task's predictability measured by the predictability filter itself: the out-of-bag pseudo-R² of a 25-tree ExtraTrees (`src/prior/filters.py` `predictability`, mirroring `repositories/TabICL.txt` `should_filter`). One row each for 120 unfiltered tasks of each prior and for the real datasets (each scored on three random 1,024-row samples, averaged). The shaded band is what the `banded` filter keeps, [0.05, 0.40]; a hollow point is a task the `tabicl` filter rejects (bootstrap p ≥ 0.05); `off` keeps everything.

**Why it matters.** A prior teaches the difficulty it generates: a prior of easy tasks teaches the model that features determine the loss almost exactly, which real LGD data does not support. The filter mode is Experiment 1's second lever; TabICLv2 rejects about a quarter of regression tasks in its first stage (`papers/2026/02_Qu_TabICLv2` §Data filtering).

**What it says.** The real LGD datasets have a median pseudo-R² of 0.42, from `axa` (0.20) and `lgd_freddie` (0.22) to `lgd_lendingclub` (0.74), three of the seven inside the band. Both priors' tasks are easier: ours has a median of 0.72, and the original prior's median is in the same region. They differ at the low end — a sizeable share of the original prior's tasks (a sixth to a quarter) carry no detectable signal and `tabicl` rejects them, while ours has none below about 0.17 and `tabicl` rejects none. `banded` keeps about one task in seven of either prior.

In [ ]:
SCORES = mechanism_plots.predictability_scores(CONFIG, REAL, n=120)

In [ ]:
FIGS.save(mechanism_plots.plot_predictability(SCORES, CONFIG), "predictability",
    caption="ExtraTrees out-of-bag pseudo-R² per task for 120 unfiltered tasks of each prior and per real LGD dataset (mean over three 1,024-row samples), with each row's median; the shaded band marks the range kept by the banded filter and hollow points the tasks rejected by the tabicl filter at p of 0.05 or more.");

### C2 · What does the model actually read?

**What it shows.** Twelve random rows of one generated task and of one real dataset (`heloc`): the seven columns with the most distinct values plus the target, each column scaled to its own range.

**Why it matters.** Summary statistics can match while the tables look nothing alike; this is the input the model reads.

**What it says.** `heloc`'s target sits almost entirely at exactly 0 or 1, and two of its columns (f6, f7) move together from row to row; the generated target mixes rows on the atoms with interior values, and its columns spread over their whole range. (The order of a generated table's columns can change between runs; its target does not.)

In [ ]:
_name, _real_one = next(iter(REAL.items())) if REAL else (None, None)
FIGS.save(e1.plot_side_by_side_tables(loaded["credit"][0], _real_one, task=TASK, real_name=_name and _name.split(".", 1)[-1]), "side_by_side_tables",
    caption="Twelve random rows of one generated credit-prior task and of one real LGD dataset as heatmaps, the seven most varied columns plus the target separated by a vertical rule, each column min-max scaled to its own range.");

### C3 · How strongly do the features depend on each other?

**What it shows.** The eigenvalues of each table's feature-correlation matrix, divided by the largest, against their rank divided by the number of columns: faint lines single tables, bold lines the medians. Generated tables are measured on the columns that vary; a real table wider than 100 columns on a random 100.

**Why it matters.** O'Prior's central realism measure: a steep fall means a few directions carry most of the variance, a flat one nearly independent columns.

**What it says.** Both priors fall steeply and almost coincide; the real LGD datasets fall far more slowly (0.1 B4). As for PD, both priors generate more strongly dependent features than real credit tables have.

In [ ]:
FIGS.save(pp.plot_spectrum_by_variant(loaded, real=REAL), "spectrum_by_variant",
    caption="Eigenvalue spectra of the feature correlation matrix, each normalised by its largest eigenvalue and plotted against eigenvalue rank over the number of varying columns, for 40 generated tasks per prior and every real LGD dataset; bold lines are medians.");

## Summary

The priors in text: the two priors and their boundary mass against the real books (A1–A3), the
distance ranking (A4), then the predictability of their tasks next to the real datasets (C1). Printed
last, so `output/All_Results.md` carries the numbers, followed by the `tfm-library` sources cited
(pin `e5ce016`) and the list of figures.

In [ ]:
print(summaries.prior_summary(loaded, TASK, source=SOURCE, reference=REFERENCE, config=CONFIG))
print()
print(summaries.realism_summary(loaded, REAL, TASK))
print()
print(mechanism_plots.predictability_summary(SCORES, CONFIG))
print()
print(literature.references_md(["outlier_clamp", "quantiles", "kumaraswamy", "oprior_scope", "crps", "filter_rate_reg", "filter_pval", "purucker_missing"]))
print()
print(FIGS.summary())